# QueryBot v0.2 — Intent-gated resolution

Demonstrates the three-step flow that QueryBot uses internally:

```
record_intent → resolve_intent → compute_metrics
```

**Part 1** (no API key): `SpecResolver` used directly to show catalog validation — exact match, typo correction, wrong dimension kind, unknown metric.

**Part 2** (requires API key): Multi-turn `QueryBot` conversation. Each turn's trace shows the three mandatory steps before any analysis or final answer.

**Part 3** (Anthropic only): Prompt-cache efficiency. After turn 1 warms Layers A and B, subsequent turns report `cache_read_tokens > 0`.

### Prerequisites

1. Set your Anthropic API key: `export ANTHROPIC_API_KEY=sk-ant-...`
2. Install the agent extra: `pip install aitaem[agent-anthropic]`
3. Set `aitaem_folder_path` below to the path of your local `aitaem` repo.

Set the path to the local folder which contains your AITAEM examples. The following sub-folders are needed inside `aitaem_folder_path`:
- `examples/data/`
- `examples/metrics/`
- `examples/slices/`
- `examples/segments/`

In [ ]:
aitaem_folder_path = "/path/to/aitaem"  # e.g. "/Users/you/Workspace/aitaem"

In [ ]:
import sys
import importlib

sys.path.insert(0, aitaem_folder_path + "/examples")
ex = importlib.import_module("03_intent_resolution_example")

## Setup — load specs and connect to DuckDB

In [ ]:
spec_cache_full, spec_cache, db_path = ex.setup(base_path=aitaem_folder_path)
print(
    f"Full catalog: {len(spec_cache_full.metrics)} metrics, "
    f"{len(spec_cache_full.slices)} slices, "
    f"{len(spec_cache_full.segments)} segment(s)"
)
print(f"  Metrics  : {', '.join(spec_cache_full.metrics)}")
print(f"  Slices   : {', '.join(spec_cache_full.slices)}")
print(f"  Segments : {', '.join(spec_cache_full.segments)}")
print(
    f"QueryBot catalog (no segments): {len(spec_cache.metrics)} metrics, "
    f"{len(spec_cache.slices)} slices"
)
print(f"DuckDB path ready: {db_path}. Connection will open in Part 2.")

---

## Part 1 — SpecResolver: catalog validation without an LLM

`SpecResolver` is the deterministic core behind `resolve_intent`.  It validates the LLM's proposed `metric_name`, `slices`, and `segment` against the catalog and returns either an `ExactMatch` or a list of `NearMiss` objects explaining what was wrong.

The four scenarios below cover the most common validation paths.

### Scenario 1 — Exact match

The LLM proposes `total_revenue` with slice `campaign_type`.  Both names exist in the catalog and `campaign_type` is registered as a slice (not a segment), so the resolver returns an `ExactMatch`.

In a live session the `resolve_intent` tool would mint a `spec_token` here (e.g. `sm_a3f1…`).  `SpecResolver` leaves it empty — token minting is the tool's job.

In [ ]:
resolver = ex.SpecResolver()

intent = ex.MetricIntent(
    metric_concept="total revenue",   # free-text — LLM's interpretation
    scope="overall",
    period_type="all_time",
)

result = resolver.resolve(
    intent,
    proposed_metric_name="total_revenue",   # LLM's catalog guess
    proposed_slices=["campaign_type"],
    proposed_segment=None,
    spec_cache=spec_cache_full,
)
ex.print_resolution("Exact match: total_revenue sliced by campaign_type", result)

### Scenario 2 — Typo: refused with user-nudge

`total_revenu` is not in the catalog.  The resolver runs `difflib.get_close_matches` (cutoff 0.75) and populates `NearMiss.suggestions` with the closest catalog name.

The LLM **does not auto-substitute** — even an obvious typo results in `status="refused"`.  Instead, the LLM includes the suggestion in its refusal message:

> *"I couldn't find a metric called `total_revenu`. Did you mean `total_revenue`?"*

The user confirms on the next turn, which then resolves to an exact match.  This conservative approach ensures no metric is silently swapped — the user always sees what is being computed.

In [ ]:
intent = ex.MetricIntent(
    metric_concept="total revenue",
    scope="overall",
    period_type="all_time",
)

result = resolver.resolve(
    intent,
    proposed_metric_name="total_revenu",   # typo: missing trailing 'e'
    proposed_slices=[],
    proposed_segment=None,
    spec_cache=spec_cache_full,
)
ex.print_resolution("Typo: 'total_revenu' → refused with user-nudge via suggestions", result)

### Scenario 3 — Wrong dimension kind

`platform` is a **segment** spec, not a slice.  Passing it in `proposed_slices` gives `why_not="wrong_dimension_kind"` — the name exists in the catalog but in the wrong category.  The LLM learns to move it to `proposed_segment` instead.

In [ ]:
intent = ex.MetricIntent(
    metric_concept="CTR",
    scope="overall",
    period_type="all_time",
)

result = resolver.resolve(
    intent,
    proposed_metric_name="ctr",
    proposed_slices=["platform"],   # 'platform' is a segment spec, not a slice
    proposed_segment=None,
    spec_cache=spec_cache_full,
)
ex.print_resolution("Wrong dimension kind: 'platform' is a segment, not a slice", result)

### Scenario 4 — Unknown metric

`profit_margin` is not defined anywhere in the catalog and has no close match (difflib cutoff 0.75), so `suggestions` is empty.  The LLM must set `status="refused"` and prompt the user to define the metric or rephrase the question.

In [ ]:
intent = ex.MetricIntent(
    metric_concept="profit margin",
    scope="overall",
    period_type="all_time",
)

result = resolver.resolve(
    intent,
    proposed_metric_name="profit_margin",
    proposed_slices=[],
    proposed_segment=None,
    spec_cache=spec_cache_full,
)
ex.print_resolution("Unknown metric: 'profit_margin' (no close catalog match)", result)

---

## Part 2 — QueryBot: three-step flow in a real conversation

Each turn in the trace must show:
1. `record_intent` — LLM records what it understood the user to be asking for
2. `resolve_intent` — deterministic catalog validation; returns `spec_token` on success
3. `compute_metrics` — executes using the opaque `spec_token`; LLM cannot tamper with compute parameters

Analysis tools (`rank_by_value`, `period_over_period`, …) appear after step 3 when the question calls for them.

In [ ]:
conn_mgr = ex.ConnectionManager()
conn_mgr.add_connection("duckdb", path=db_path)

bot = ex.QueryBot(
    model=ex.MODEL,
    spec_cache=spec_cache,
    connection_manager=conn_mgr,
)

### Turn 1 — multi-metric question

Two metrics in one question → two `record_intent` + `resolve_intent` + `compute_metrics` triplets, then a summary narrative.  This turn also **warms Layers A and B** in Anthropic's prompt cache.

In [ ]:
response1 = await bot.chat("What was total revenue and ROAS across all campaigns?")
ex.print_response(1, "What was total revenue and ROAS across all campaigns?", response1)
ex.print_sample(response1)
ex.print_trace(response1.trace)

### Turn 2 — breakdown with analysis

A breakdown question uses `rank_by_value` to surface the top campaign type after `compute_metrics` returns the full distribution.

In [ ]:
response2 = await bot.chat("Which industry had the highest CTR?")
ex.print_response(2, "Which industry had the highest CTR?", response2)
ex.print_sample(response2)
ex.print_trace(response2.trace)

### Turn 3 — monthly trend

A `period_type="monthly"` question; `compute_metrics` returns one row per month and the LLM narrates the trend directly (no analysis tool needed for time series).

In [ ]:
response3 = await bot.chat("How did total revenue change month-by-month in 2024?")
ex.print_response(3, "How did total revenue change month-by-month in 2024?", response3)
ex.print_sample(response3)
ex.print_trace(response3.trace)

---

## Part 3 — Prompt-cache efficiency (Anthropic only)

QueryBot v0.2 divides the system prompt into three layers:

| Layer | Content | Cached? |
|-------|---------|--------|
| A | Workflow rules and tool guidance (identical for all tenants) | Yes — `anthropic_cache_instructions="5m"` |
| B | Per-tenant metric/slice/segment catalog | Yes — same cache breakpoint |
| C | Today's date (changes each calendar day) | No — dynamic instruction |

After turn 1, Layers A+B are stored in Anthropic's prompt cache.  Turns 2 and 3 should read them from cache, reducing both latency and input token cost.  The metric is `cache_read_tokens` in the usage block.

In [ ]:
if "anthropic:" in bot._model:
    print("Cache efficiency summary (Anthropic only)")
    print("─" * 50)
    ex.cache_summary("Turn 1 (warms cache — no read expected)", response1)
    ex.cache_summary("Turn 2 (Layers A+B should be cached)", response2)
    ex.cache_summary("Turn 3 (Layers A+B should be cached)", response3)
else:
    print(f"Cache check skipped — provider is not Anthropic (model={bot._model!r}).")